# Camera images pre processing

The objective of this code is to prepare the images for the alignment and stacking of the multiple acquisitions.

The following operations are performed in this file:

* Import of the pictures.
* Structuring of imported data.
* Conversion into floating point encoding.
* Quality scoring of the images.
* Channelwise pre-processing.

The data will be saved in the form of a Pickle file, to preserve the python compatible structure, to be easily used in future python codes.

The following libraries will be used:

* Notice that some functions are bespoke and come from self owned code.
* A specific Functions library has been built to have a slimmer code.

In [4]:
# External python packages
import numpy as np
import matplotlib.pyplot as plt
import glob
import cv2
import astroalign as aa
import os
import rawpy
from pathlib import Path
from PIL import Image
import gc
import imageio.v3 as iio
from skimage.restoration import richardson_lucy
from typing import Any
import pickle

# Cross references to other function libraries
from Functions_library import Conversion_functions as Conversions
from Functions_library import Import_functions as Imports

## Input Parameters setup and initialization

Here we want to set all the user defined variables in order to leave the rest of the code to the actual processing.

In [ ]:
camera_acquisitions_folder = "C:/Users/filip/Desktop/Sessione_25-12-28/Orion 01/Foto all'ombra"
midsave_folder = "C:/Users/filip/Desktop/Sessione_25-12-28/Orion 01/Foto all'ombra/MidSaves"
input_format = ".dng" # Da usare se si vuole importare un formato nello specifico
output_format = ".tif" # Formato di output desiderato
processing_format = "float32" # Formato di elaborazione desiderato: "uint8", "uint16", "float32", "float64"
output_bit_depth = "uint16" # Profondità di bit di output desiderata: "uint8", "uint16", "float32"

life_in_the_fast_lane = True

## Initialization

The data is structured here to give a clear order and a tidy frame to store everything.

In particular we want to tag every image with the place it takes inside the whole list.
The single iamge will be characterized by:

* The File name 
* Its colored image, with the 3 BGR channels
* Its black and white image, used for the alignments
* All the transformations it underwent
* Some quality scoring

To keep the code fluid the single variables will be declared gradually

## Images Import

We will now actually import the images contained inside the specified folder.

The data structure can already be understood and the import result is stored inside a mid-save variable.

In [ ]:
database = []
for file in Path(camera_acquisitions_folder).glob(f'*{input_format}'):
    try:
        data_element: dict[str, Any] = { 'file_path' : str(file) ,
                                         'bgr_image' : None      }
        data_element['bgr_image'] = Imports.general2bgr(str(file), processing_format)
        database.append(data_element)
        print(f"✅ Caricata immagine numero {len(database)}: {file}.")
    except Exception as e:
        print(f"❌ Errore nel caricamento dell'immagine {file}: {e}")
print(f"✅ Caricate {len(database)} immagini.")

if not life_in_the_fast_lane:
    # Salvataggio intermedio del database
    os.makedirs(midsave_folder, exist_ok=True)
    midasave_database = f"{midsave_folder}/Imported_images_database.pkl"
    with open(midasave_database, 'wb') as f:
        pickle.dump(database, f)
    print(f"✅ Database salvato in {midasave_database}.")

✅ Caricata immagine numero 1: C:\Users\filip\Desktop\Sessione_25-12-28\Orion 01\Foto all'ombra\Lights_0001_20251228_234541.dng.
✅ Caricata immagine numero 2: C:\Users\filip\Desktop\Sessione_25-12-28\Orion 01\Foto all'ombra\Lights_0002_20251228_234552.dng.
✅ Caricata immagine numero 3: C:\Users\filip\Desktop\Sessione_25-12-28\Orion 01\Foto all'ombra\Lights_0003_20251228_234604.dng.
✅ Caricata immagine numero 4: C:\Users\filip\Desktop\Sessione_25-12-28\Orion 01\Foto all'ombra\Lights_0004_20251228_234616.dng.
✅ Caricata immagine numero 5: C:\Users\filip\Desktop\Sessione_25-12-28\Orion 01\Foto all'ombra\Lights_0005_20251228_234628.dng.
✅ Caricata immagine numero 6: C:\Users\filip\Desktop\Sessione_25-12-28\Orion 01\Foto all'ombra\Lights_0006_20251228_234639.dng.
✅ Caricata immagine numero 7: C:\Users\filip\Desktop\Sessione_25-12-28\Orion 01\Foto all'ombra\Lights_0007_20251228_234657.dng.
✅ Caricata immagine numero 8: C:\Users\filip\Desktop\Sessione_25-12-28\Orion 01\Foto all'ombra\Lights_00

### Channels alignment

Now that we have all the images uploaded we want to start refining them.

The first step of data processing is to perferctly align the single color channels to have every star perfectly aligned with itself.